In [48]:
import pandas as pd
import numpy as np

movies_df = pd.read_csv('data/movies.csv')
movies = movies_df.copy()

ratings_df = pd.read_csv('data/ratings.csv')
ratings = ratings_df.copy()

tags_df = pd.read_csv('data/tags.csv')
tags = tags_df.copy()

Filtering ratings. Bad movies and impolular movies are excluded

In [49]:
nup_movies = movies.drop_duplicates("title")
movies_valid_genres = nup_movies[nup_movies["genres"] != "(no genres listed)"]



In [ ]:
lowest_acceptable_mean_rating_score = 1.5 # Set to zero to disable filtering
lowest_acceptable_rating_count = 50 # Set to zero to disable filtering

#ratings = ratings.drop(columns= ["timestamp"])

ratings_grouped = ratings.groupby('movieId')['rating'].mean().sort_values()
good_movie_scores = ratings_grouped[ratings_grouped >= lowest_acceptable_mean_rating_score].index
ratings_good = ratings[ratings["movieId"].isin(good_movie_scores)]

ratings_per_movie = ratings.groupby('movieId')['rating'].count()
popular_movie = ratings_per_movie[ratings_per_movie >= lowest_acceptable_rating_count].index
ratings_good_and_popular = ratings_good[ratings_good["movieId"].isin(popular_movie)]


ratings_with_name = pd.merge(ratings_good_and_popular, movies_valid_genres, how='inner', on='movieId')
ratings_with_name.shape[0] / ratings.shape[0]


0.7026906232004919

In [ ]:
lowest_acceptable_mean_rating_score = 1.5 # Set to zero to disable filtering
lowest_acceptable_rating_count = 50 # Set to zero to disable filtering

ratings_grouped = ratings.groupby('movieId')['rating'].mean().sort_values()
good_movie_scores = ratings_grouped[ratings_grouped >= lowest_acceptable_mean_rating_score].index
ratings_good = ratings[ratings["movieId"].isin(good_movie_scores)]

ratings_per_movie = ratings.groupby('movieId')['rating'].count()
popular_movie = ratings_per_movie[ratings_per_movie >= lowest_acceptable_rating_count].index
ratings_good_and_popular = ratings_good[ratings_good["movieId"].isin(popular_movie)]
##

ratings_per_movie = ratings.groupby('movieId')['rating'].agg(['mean', 'count'])

good_movies = ratings_per_movie[
    (ratings_per_movie['mean'] >= 1.5) &
    (ratings_per_movie['count'] >= 50)
].index

nup_movies_good = nup_movies[nup_movies["movieId"].isin(good_movies)].reset_index(drop=True)
##
ratings_good_movies = ratings.merge(nup_movies_good, on="movieId")
ratings_good_movies.drop(["timestamp"], axis=1, inplace=True)

nup_movies_diff = nup_movies_good.shape[0] / nup_movies.shape[0]
ratings_diff =  ratings_good_movies.shape[0] / ratings.shape[0]
ratings_diff


Splitting movie genres string with on-hot encoder and excluding movies without genre

In [ ]:
genre_dummies = movies["genres"].str.get_dummies("|")
movies_genre_one_hot = pd.concat([movies, genre_dummies], axis=1)
movies_filtered = movies_genre_one_hot[movies_genre_one_hot["(no genres listed)"] == 0]
movies_filtered = movies_filtered.drop(columns= ["(no genres listed)"])



Filtering bad and impopular movies from movies

In [ ]:
ratings_good_and_popular_list = ratings_good_and_popular["movieId"].unique()
movies_filtered = movies_filtered[movies_filtered["movieId"].isin(ratings_good_and_popular_list)]

movies_filtered = movies_filtered.reset_index(drop=True)
all_movies_genre_matrix = movies_filtered.drop(columns=["movieId", "title", "genres"])



TF-IDF: Movie genres and user tags are combined as text for each movie as preparation for TF-IDF

In [ ]:
movies_tags_TF_IDF = movies_filtered[["movieId", "title", "genres", "genres_text"]]
movies_tags_TF_IDF["genres_text"] = movies_tags_TF_IDF["genres"].str.replace("|", " ", regex=False)
tags_filtered = tags[tags["movieId"].isin(movies_tags_TF_IDF["movieId"])] # Removing rows related to movies that have been filtered
tags_filtered = tags_filtered.dropna(subset=["tag"])                              # Remove NaN

tags_grouped = tags_filtered.groupby("movieId")["tag"].apply(lambda x: " ".join(x)).reset_index() # Group all movie tags for each film
movies_tags_merged = movies_tags_TF_IDF.merge(tags_grouped, on="movieId", how="left") # Merge movies with tags, even if movie lack tags 
movies_tags_merged["tag"] = movies_tags_merged["tag"].fillna("")

movies_tags_merged["combined_text"] = (
    movies_tags_merged["genres_text"] + " " + movies_tags_merged["tag"])
movies_tags_merged

,movieId,title,genres,genres_text,tag,combined_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy,animation friendship toys animation Disney Pix...,Adventure Animation Children Comedy Fantasy an...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy,animals based on a book fantasy magic board ga...,Adventure Children Fantasy animals based on a ...
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance,sequel moldy old old age old men wedding old p...,Comedy Romance sequel moldy old old age old me...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance,characters chick flick girl movie characters c...,Comedy Drama Romance characters chick flick gi...
4,5,Father of the Bride Part II (1995),Comedy,Comedy,family pregnancy wedding 4th wall aging baby d...,Comedy family pregnancy wedding 4th wall aging...
...,...,...,...,...,...,...
15994,287377,Fast X (2023),Action|Crime,Action Crime,action bomb car chase duringcreditsstinger esc...,Action Crime action bomb car chase duringcredi...
15995,287633,Asteroid City (2023),Comedy|Drama|Romance|Sci-Fi,Comedy Drama Romance Sci-Fi,boring 1950s 1955 astronomy children comedy-dr...,Comedy Drama Romance Sci-Fi boring 1950s 1955 ...
15996,288167,Extraction 2 (2023),Action|Thriller,Action Thriller,action betrayal brothers child in danger child...,Action Thriller action betrayal brothers child...
15997,288265,Mission: Impossible - Dead Reckoning Part One ...,Action|Adventure|Thriller,Action Adventure Thriller,action action packed Christopher McQuarrie Mis...,Action Adventure Thriller action action packed...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies_tags_merged["combined_text"])
cosine_sim_tfidf = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim_tfidf


array([[1.        , 0.06024607, 0.01940326, ..., 0.01226778, 0.05686285,
        0.17743418],
       [0.06024607, 1.        , 0.00458691, ..., 0.01969276, 0.01149181,
        0.04954591],
       [0.01940326, 0.00458691, 1.        , ..., 0.01898711, 0.        ,
        0.00581708],
       ...,
       [0.01226778, 0.01969276, 0.01898711, ..., 1.        , 0.07197562,
        0.10798481],
       [0.05686285, 0.01149181, 0.        , ..., 0.07197562, 1.        ,
        0.04920666],
       [0.17743418, 0.04954591, 0.00581708, ..., 0.10798481, 0.04920666,
        1.        ]], shape=(15999, 15999))

TF-IDF

In [ ]:
def recommend_TFIDF(movie_title, n=5):
    matches = movies_tags_merged[movies_tags_merged["title"] == movie_title]
    if matches.empty:
       return "Movie not found"
    
    idx = matches.index[0]
    input_movie_genre = tfidf_matrix[idx]

    # Inputfilm vs all movies -> Similarity
    similarity_scores = cosine_similarity(input_movie_genre, tfidf_matrix).flatten()

    similarity_scores_df = pd.DataFrame({"similarity_score_%": 100*similarity_scores}) # Array from cosine_similarity to pandas dataframe
    similarity_scores_sorted_df = similarity_scores_df.sort_values(by="similarity_score_%", ascending=False)
    similarity_scores_sorted_top5_df = similarity_scores_sorted_df.iloc[1:n+1]   # Removing self
    similarity_df = similarity_scores_sorted_top5_df                             # shortening name

    similarity_df["title"] = movies_tags_merged.loc[similarity_df.index, "title"]  # Replaces movieId with title 
    similarity_df = similarity_df.loc[:, ['title', 'similarity_score_%']]         # Changing order of columns
    return similarity_df
recommend_TFIDF("Yellow Submarine (1968)", 5)

,title,similarity_score_%
9032,Magical Mystery Tour (1967),81.020943
2657,Help! (1965),77.719072
9230,Across the Universe (2007),70.786918
2577,"Hard Day's Night, A (1964)",66.796355
4070,Imagine: John Lennon (1988),60.603871


Rekommender Cosine
"identify similarities between movies based on genres based filtering KNN och cosine similarity"
"one-hot encoding on genres and a KNN-Transform with cosine similarity"

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_Cosine(movie_title, n=5):
    matches = movies_filtered[movies_filtered["title"] == movie_title]
    if matches.empty:
       return "Movie not found"
    
    idx = matches.index[0]
    input_movie_genre = all_movies_genre_matrix.iloc[idx].values.reshape(1, -1)

    # Inputfilm vs all movies -> Similarity
    similarity_scores = cosine_similarity(input_movie_genre, all_movies_genre_matrix).flatten()

    similarity_scores_df = pd.DataFrame({"similarity_score_%": 100*similarity_scores}) # Array from cosine_similarity to pandas dataframe
    similarity_scores_sorted_df = similarity_scores_df.sort_values(by="similarity_score_%", ascending=False)
    similarity_scores_sorted_top5_df = similarity_scores_sorted_df.iloc[1:n+1]   # Removing self
    similarity_df = similarity_scores_sorted_top5_df                             # shortening name

    similarity_df["title"] = movies_filtered.loc[similarity_df.index, "title"]  # Replaces movieId with title 
    similarity_df = similarity_df.loc[:, ['title', 'similarity_score_%']]         # Changing order of columns
    return similarity_df
recommend_Cosine("Yellow Submarine (1968)", 5)


,title,similarity_score_%
6890,"Chipmunk Adventure, The (1987)",91.287093
12089,Frozen (2013),91.287093
6410,Allegro non troppo (1977),89.442719
6957,Into the Woods (1991),89.442719
15547,Adventure Time: Islands (2017),89.442719


In [ ]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(all_movies_genre_matrix)

def recommend_KNN(movie_title, n=5):
    matches = movies_filtered[movies_filtered["title"] == movie_title]
    if matches.empty:
       return "Movie not found"

    idx = matches.index[0]
    input_movie_genre = all_movies_genre_matrix.iloc[[idx]]

    knn_distances, knn_indices = knn.kneighbors(input_movie_genre, n_neighbors=n+1)
    # platta arrays
    indices = knn_indices.flatten()[1:]      # hoppa över sig själv
    distances = knn_distances.flatten()[1:]

    result = movies_filtered.iloc[indices][["title"]].copy()

    similarity_scores = 100 - 100*distances    # cosine distance → similarity
    result["similarity_score_%"] = similarity_scores

    return result

recommend_KNN("Yellow Submarine (1968)", 5)


,title,similarity_score_%
6890,"Chipmunk Adventure, The (1987)",91.287093
12089,Frozen (2013),91.287093
15890,Luck (2022),89.442719
6410,Allegro non troppo (1977),89.442719
9116,How the Grinch Stole Christmas! (1966),89.442719
